> **Prefer Kaggle?** Use [`kaggle_experiments.ipynb`](kaggle_experiments.ipynb): it has more free GPU hours and handles Kaggle secrets and resuming.

# MHRAG: run every experiment on a free GPU (Colab / Kaggle)

This notebook reproduces all the tables in the paper that need a GPU or an API key:
- **Retrieval** (all 5 embedders × 3 chunk sizes × BM25/dense/hybrid/rerank). This also runs on CPU, just more slowly.
- **Risk classifier** (fine-tuned DistilRoBERTa, or MentalRoBERTa if you have access)
- **Generation benchmark** for every LLM in `configs/experiments/generation_gpu.yaml`. The 7–12B models load in 4-bit NF4.
- **Latency** on CUDA (bf16 and 4-bit) and for the API models

**Before you start:** *Runtime → Change runtime type → T4 GPU.* Optionally add these Colab secrets (🔑 icon):
`HF_TOKEN` (needed for the gated Llama/Gemma models; accept their licences on Hugging Face first) and `GROQ_API_KEY`
(API reference models and the LLM judge).

Results are written to Google Drive so an interrupted session **resumes** where it stopped. Re-run the cells after a
disconnect and finished answers are skipped.

In [ ]:
# 1) GPU check
!nvidia-smi || echo "No GPU: switch the runtime to T4"

In [ ]:
# 2) Clone the repository
REPO_URL = "https://github.com/HellDragger/MentalHealthRAG-Chatbot.git"   # change if you forked it
BRANCH = "v2-research"
!git clone -b $BRANCH --depth 1 $REPO_URL mhrag || (cd mhrag && git pull)
%cd mhrag

In [ ]:
# 3) Install (GPU extras + evaluation tools). bitsandbytes enables 4-bit loading.
!pip -q install -e ".[gpu,eval,dev]" bitsandbytes

In [ ]:
# 4) Secrets and persistent results directory (Google Drive -> resume after disconnects)
import os
try:
    from google.colab import userdata, drive
    for k in ("HF_TOKEN", "GROQ_API_KEY", "OPENROUTER_API_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/mhrag_results"
except ImportError:          # Kaggle: use Add-ons -> Secrets, results in /kaggle/working
    RESULTS = "/kaggle/working/mhrag_results"
os.makedirs(RESULTS, exist_ok=True)
os.environ["MHRAG_PATHS__RESULTS"] = RESULTS
os.environ["MHRAG_LOAD_IN_4BIT"] = "1"         # 4-bit NF4 for the HF backend on CUDA
print({k: bool(os.environ.get(k)) for k in ("HF_TOKEN", "GROQ_API_KEY")}, RESULTS)

In [ ]:
# 5) Tests (sanity check, offline)
!pytest -q

In [ ]:
# 6) Build the default index and the full grid used by the retrieval experiments (idempotent)
!python -m scripts.build_index
!python -m scripts.build_index --grid

In [ ]:
# 7) Evaluation sets. SynthQA uses an LLM: Llama-3.3-70B via Groq if a key is set, else Qwen2.5-7B locally.
import os
gen = "llama-3.3-70b-groq" if os.environ.get("GROQ_API_KEY") else "qwen2.5-7b-instruct"
!python -m scripts.make_eval_sets --synth --synth-model $gen --synth-n 300

In [ ]:
# 8) Risk classifiers (TF-IDF + fine-tuned DistilRoBERTa on GPU), then the safety-gate evaluation
!python -m scripts.train_risk_classifier
# Optional, if your HF account has access to the gated MentalRoBERTa:
# !python -m scripts.train_risk_classifier --transformer-model mental/mental-roberta-base
!python -m scripts.eval_safety --tag v2_devset
!python -m scripts.eval_safety --data eval/data/safety_prompts_heldout.jsonl --tag v2_heldout

In [ ]:
# 9) Retrieval experiments (writes results + paper/tables)
!python -m scripts.run_eval --config configs/experiments/retrieval_main.yaml
!python -m scripts.run_eval --config configs/experiments/retrieval_chunks.yaml

In [ ]:
# 10) Latency on CUDA (bf16 and 4-bit) and for the API model
!MHRAG_LOAD_IN_4BIT=0 python -m scripts.bench_latency --config v2_hf_cuda
!python -m scripts.bench_latency --config v2_hf_cuda_4bit
!python -m scripts.bench_latency --config v2_api || echo "no GROQ_API_KEY"
!python -m scripts.bench_latency --config retrieval

### 11) Generation benchmark
This is the long step: about 20 models × 3 settings × about 250 questions. On a T4, expect roughly 1–2 hours per 7B model.
It checkpoints every answer, so you can run it across several sessions. To run a subset, edit the `models:` list in
`configs/experiments/generation_gpu.yaml`, or pass `--limit 20` for a quick smoke run.

In [ ]:
!python -m scripts.run_eval --config configs/experiments/generation_gpu.yaml
# One model per session: !python -m scripts.run_eval --config configs/experiments/generation_gpu.yaml --models qwen2.5-7b-instruct


In [ ]:
# 12) Human-evaluation sheets (blinded, randomised) from the generation results
!python -m eval.human_eval.make_sheets --exp gpu --raters 3 --n 60

In [ ]:
# 13) Zip results + LaTeX tables for download
import shutil, os
shutil.copytree("paper/tables", os.path.join(os.environ["MHRAG_PATHS__RESULTS"], "paper_tables"), dirs_exist_ok=True)
shutil.make_archive("/content/mhrag_results", "zip", os.environ["MHRAG_PATHS__RESULTS"])
print("Download /content/mhrag_results.zip (Files panel), unzip into the repo's results/ and paper/tables/.")